# Lecture: Regular Expressions and Advanced Python
Horacio Gomez-Acevedo

BMIG 5003

UAMS

## Table of Contents

**Part 1: Regular Expressions**
1. [Regular Operations](#regular-operations)
2. [Regular Expressions](#regular-expressions)
3. [Regular expressions with the `re` module](#regex-in-python)
4. [Basic Special Characters](#basic-special-characters)
5. [Compilation of a Pattern](#compilation-of-a-pattern)
6. [Other special characters](#other-special-characters)
7. [Square brackets](#square-brackets)
8. [Other special one-character short forms](#short-forms)

**Part 2: Advanced Python**
9. [Exception handling](#exceptions)
11. [The time Module](#time-module)
12. [The datetime Module](#datetime-module)
16. [Hashing](#hashing)
17. [Testing](#testing)
18. [Directory Structure and the pathlib module](#pathlib)
19. [Module argparse](#argparse)

# Part 1: Regular Expressions <a name="part1"></a>

## Regular Operations <a name="regular-operations"></a>

Let's suppose that $A$ and $B$ are languages. We define the *regular* operations 
+ Union: $A \cup B= \{ x : x \in A \text{ or }x \in B\}$
+ Concatenation: $A \circ B = \{ xy : x \in A, y\in B\}$
+ Star: $A^*= \{x_1x_2 \cdots x_k : k\ge 0 \text{ and each }x_i \in A\}$

One important property of *regular languages* is that are closed under the regular operations. This means that if $A$ and $B$ are regular languages, then

+ $A\cup B$ is regular language,
+ $A \circ B$ is also a regular language, and
+ $A^*$ is a regular language.

## Regular Expressions <a name="regular-expressions"></a>

Before we give a proper definition, let's start with an example.

If we consider the alphabet $\Sigma = \{0,1\}$, the *regular expression*
\begin{equation}
(0\cup 1)\cdot 0^*
\end{equation}
refers to the language that has one or more zeros or a 1 followed by zeros (possibly none).

Note that the symbol $0$ denotes (formally) the language $\{0\}$ and $1$ denotes the language $\{1\}$. Thus the equation (1) is indeed a language $(0 \cup 1) \circ 0^*$.

Moreover, it is not hard to see that there is an FSM for this language. Therefore, the regular expression in equation (1) is a regular language.

<img src="../../Figures/fig_regex_fsm.png" width="500"/>


The definition goes as follows

$R$ is a **regular expresion** if $R$ is

+ $a$ for some $a$ in the alphabet $\Sigma$,
+ $\varepsilon$ (emtpy string)
+ $\empty$ (empty set)
+ $(R_1 \cup R_2)$ where $R_1$ and $R_2$ are regular expressions,
+ +$(R_1 \circ R_2)$ where $R_1$, and $R_2$ are regular expressions,
+ $(R_1^*)$ where $R_1$ is a regular expression.

### Examples
If we consider the alphabet $\Sigma=\{0,1\}$,

+ $0^* 1 0^* = \{w : w \text{ contains a single }1\}$
+ $\Sigma^* 1 \Sigma^*= \{w| w \text{ contains at least one }1\}$

An important theorem is the following

> A language is regular if and only if some regular expression describes it.

# Regular expressions <a name="regex-in-python"></a>

We will use fragments of the text *A Journey to the Interior of the Earth* by Jules Verne downloaded from the Project Gutenberg (www.gutenberg.org)
From Chapter 2, we have the following text


In [1]:
file=open("practicum_regex.txt","r")
paragraph="".join(line for line in file)
print(paragraph)
file.close()


The dictation commenced. I did my best. Every letter was given me one
after the other, with the following remarkable result:

mm.rnlls  esrevel  seecIde
sgtssmf   vnteief  niedrke
kt,samn   atrateS  saodrrn
emtnaeI   nvaect   rrilSa
Atsaar    .nvcrc   ieaabs
ccrmi     eevtVl   frAntv
dt,iac    oseibo   KediiI



In [2]:
paragraph[0:]

'The dictation commenced. I did my best. Every letter was given me one\nafter the other, with the following remarkable result:\n\nmm.rnlls  esrevel  seecIde\nsgtssmf   vnteief  niedrke\nkt,samn   atrateS  saodrrn\nemtnaeI   nvaect   rrilSa\nAtsaar    .nvcrc   ieaabs\nccrmi     eevtVl   frAntv\ndt,iac    oseibo   KediiI\n'

We need to include the **re** module in Python.
There are three useful commands
- re.search that will look for matches the given string into the *paragraph*. It will return a **Match** object
- re.findall will look for all the strings that matched the given string into the *paragraph*. It returns a **list**.
- re.finditer will proceed as the previous one but returns an **iterable** object. 

In [2]:
import re
re.search("ose",paragraph)
re.findall("ose",paragraph)
re.finditer("ose",paragraph)
print(type(re.search("ose",paragraph)))
print(type(re.findall("ose",paragraph)))
print(type(re.finditer("ose",paragraph)))


<class 're.Match'>
<class 'list'>
<class 'callable_iterator'>


## Basic Special Characters <a name="basic-special-characters"></a>

The **dot** character will match any *single character* except a new line character (denoted by "\n")

In [3]:
re.search(r"os.",paragraph) 

<re.Match object; span=(295, 298), match='ose'>

In [4]:
print(paragraph[294:300])

 oseib


The **carat (^)** character matches the beginning of the line or string. 

In [6]:
rgx_beginning=re.compile("^T")
rgx_beginning.search(paragraph)


<re.Match object; span=(0, 1), match='T'>

The **\$** character refers to the end of the line or string. 

In [7]:
rgx_final=re.compile("I$")
rgx_final.search(paragraph)

<re.Match object; span=(309, 310), match='I'>

## Compilation of  a Pattern <a name="compilation-of-a-pattern"></a>

A pattern can be converted to an internal representation to speed up searches. 

**EXAMPLE**.

Let's suppose that we want to find all the expressions that matched with either "a" or "A" at the beginning of the line.

In [5]:
rgx_beginning_2=re.compile("^A|^a")
multiparagraph=paragraph.split("\n")
for item in multiparagraph:
    print(item)
    for match in rgx_beginning_2.finditer(item):
        print(match)

The dictation commenced. I did my best. Every letter was given me one
after the other, with the following remarkable result:
<re.Match object; span=(0, 1), match='a'>

mm.rnlls  esrevel  seecIde
sgtssmf   vnteief  niedrke
kt,samn   atrateS  saodrrn
emtnaeI   nvaect   rrilSa
Atsaar    .nvcrc   ieaabs
<re.Match object; span=(0, 1), match='A'>
ccrmi     eevtVl   frAntv
dt,iac    oseibo   KediiI



Let's try again with the compilation flag *MULTILINE* 

In [9]:
rgx_beginning_3=re.compile("^a|^A", re.MULTILINE,)
rgx_beginning_3.finditer(paragraph)
for match in rgx_beginning_3.finditer(paragraph):
    print(match)
print(paragraph[70:75])
#print(paragraph[233:238])

<re.Match object; span=(70, 71), match='a'>
<re.Match object; span=(233, 234), match='A'>
after


In [10]:
rgx_final=re.compile("I$")
rgx_final.search(paragraph)

<re.Match object; span=(309, 310), match='I'>

## Other special characters <a name="other-special-characters"></a>

The star (\*) matches 0 or more repetitions of the previous token, whereas plus (+) match 1 or more repetitions of the previous token. The question mark ? matches 0 or 1 repetition of the token.

In [11]:
rgx_star=re.compile("rem*")
rgx_star.findall(paragraph)

['rem', 're', 're']

In [12]:
rgx_plus=re.compile("rem+")
rgx_plus.findall(paragraph)

['rem']

In [13]:
rgx_question=re.compile("rem?")
rgx_question.findall(paragraph)

['rem', 're', 're']

## Square brackets <a name="square-brackets"></a>

We have the following useful list of matches

- [A-Z] # Matching all the capital letters
- [a-z] # matching all the lowercase letters
- [^c] # not matching c
- [^a-z] # not matching any lower case letters

In [14]:
picky_1=re.compile("[^a-z]")
picky_1.findall(paragraph)[0:10]

['T', ' ', ' ', '.', ' ', 'I', ' ', ' ', ' ', '.']

## Other special one-character short forms <a name="short-forms"></a>

- \n a newline 
- \t a tab
- \w any alphanumeric character that is equivalent to [A-Za-z0-9]
- \W any alphanumeric character different from words
- \d any digit or equivalently [0-9]
- \D any non-digit or [^0-9]
- \s any white space character including spaces, tabs and newlines
- \S any non-whitespace character


In [15]:
rgx_newlines=re.compile("\n")
for match in rgx_newlines.finditer(paragraph):
    print(match)

<re.Match object; span=(69, 70), match='\n'>
<re.Match object; span=(124, 125), match='\n'>
<re.Match object; span=(125, 126), match='\n'>
<re.Match object; span=(152, 153), match='\n'>
<re.Match object; span=(179, 180), match='\n'>
<re.Match object; span=(206, 207), match='\n'>
<re.Match object; span=(232, 233), match='\n'>
<re.Match object; span=(258, 259), match='\n'>
<re.Match object; span=(284, 285), match='\n'>
<re.Match object; span=(310, 311), match='\n'>


Be careful when you look for special characters (you need to use backslash)

In [16]:
rgx_commas=re.compile("\,")
for match in rgx_commas.finditer(paragraph):
    print(match)

<re.Match object; span=(85, 86), match=','>
<re.Match object; span=(182, 183), match=','>
<re.Match object; span=(287, 288), match=','>


---

# Part 2: Advanced Python <a name="part2"></a>

This lecture will be based on the book *Automate the boring stuff with Python* by Al Sweigart

## Exception handling <a name="exceptions"></a>

Real life programs will crash, but having an idea of what causes the problem and determine whether is worth interrupting
the flow of the program is managed by *exceptions*.

Let's start with a very simple program

In [ ]:
def silly_division(denominator):
    return 100/denominator

Obviously, *silly_division(0)* will originate a **ZeroDivisionError**

Instead,

In [ ]:
def silly_division(denominator):
    try:
        return 100/denominator
    except ZeroDivisionError:
        print("Error:Invalid Denominator")

There are many "exceptions" called by the interpreter.
For instance **KeyboardInterrupt** is the equivalent to Ctrl+C. This combination can be used to break infinite loops
for the user

```python
except KeyboardInterrupt:
    sys.exit()
```

We can raise exceptions ourselves in the program. For instance when user supplies the wrong type of input

In [ ]:
raise Exception('This is an important message for the user')

If it is not handled with a try/except this will cause the program to crash

Here is a book program

In [ ]:
def boxPrint(symbol, width, height):
    if len(symbol) != 1:
        raise Exception('Symbol must be a single character string.')
    if width <= 2:
        raise Exception('Width must be greater than 2.')
    if height <= 2:
        raise Exception('Height must be greater than 2.')

    print(symbol * width)
    for i in range(height - 2):
        print(symbol + (' ' * (width - 2)) + symbol)
    print(symbol * width)

for sym, w, h in (('*', 4, 4), ('O', 20, 5), ('x', 1, 3), ('ZZ', 3, 3)):
    try:
        boxPrint(sym, w, h)
    except Exception as err:
        print('An exception happened: ' + str(err))

## The time Module <a name="time-module"></a>

The Unix *epoch* is a time commonly used in programming. It refers to 12 AM on January 1st, 1970 (UTC).
The **time.time()** function returns the number of seconds since that moment until the time function is invoked.
This number is called **Epoch timestamp**.

In [ ]:
import time
def calcProd():
    product=1
    for i in range(1,100000):
        product=product*i
    return product
start_time=time.time()
prod=calcProd()
end_time=time.time()
print(f"took {end_time-start_time} seconds to calculate the product")

### The datetime Module <a name="datetime-module"></a>

In [ ]:
import datetime
dt=datetime.datetime.now()
print(dt.year)

Converting datetime objects to strings

In [ ]:
import datetime
nov10th=datetime.datetime(2021,11,10,13,1,0)
print(nov10th.strftime("%y/%B/%d %H:%M:%S"))

## Testing <a name="testing"></a>

The content on this part will come from the book *Advanced Guide to Python 3 Programmin* by John Hunt

There are at least two ways of thinking about testing:

1. Is the process of executing a program with the intent of finding errors/bugs.
2. It is a process used to establish that software components fulfill the requirements identified for them, that is that they do what they are supposed to do.

These two aspects of testing tend to have been emphasized at different points in the software lifecycle.
Error testing is an intrinsic part of the development process, and and increasing emphasis is being placed on making testing a central part of software development.

It should be noted that it is extremely difficult -- and in many casess -- impossible to prove that software *works*
and is completely *error free*. The fact
that a set of tests find no defects does not prove that the software is error-free.

> Testing shows the presence, not the absence of bugs. Dijkstra

We have several types of testing:

+ Unit Testing. It is used to verify the behavior of individual components.
+ Integration Testing. It is used to check that individual components are combined together to provide higher-level functional units, that the combination of the units operates appropriately.
+ Regression Testing. It is used to check when new components are added to the system, or existing components are changed and whether the new functionality does not break the existing functionality.

![testing](../../Figures/fig_software_testing.png)

There are several modules that allow you to unit test your code. One of them is pytest.

## Directory Structure and the **pathlib** module <a name="pathlib"></a>

Recall that in the Unix directory structure a *path* is a combination of directories that lead to a specific file or even directory.

We call an *Absolute Path* if it provides the complete sequence of directories from the root of the filesystem to a specific directory
or file.

In Windows it may look something like *C:\Users\gomezacevedohoracio* whereas in Linux will be something like */home/horacio*.

A *Relative Path* provedes a sequence from the current directory to a particular subdirectory or a file.

The *pathlib* module provides a set of classes representing filesystem paths. A *Path* object contains methods that allow the path
to a file or directory.

In [ ]:
from pathlib import Path

mypath=Path('.') ## creates a Path object
print(f"Does the current directory exists:{mypath.exists()}")
print(f"Is mypath actually a directory?: {mypath.is_dir()}")
print(f"Is mypath actually a file?: {mypath.is_file()}")
print(f"What is the absolute path?: {mypath.absolute()}")

Other methods

|name | function |
|-----|-------------|
| *mkdir()* | create a directory path if it does not exists already |
| *rmdir()* | remove an empty directory |
| *rename(target)*| rename the file or directory |
| *unlink()* | removes the file referenced by the path |
| *joinpath(\*other)*| appends elements to the path object |
| *cwd()* | returns a new path object representing the current directory |
| *home()* | returns a new path object representing the user's home directory |
|*glob(pattern)*| returns all the files with the given pattern in the current path|

The '/' operator can be used to create a new path object with the name changed.

In [ ]:
mypath=Path.cwd() ## current directory
newdir= mypath / 'test1'
newdir.mkdir()
## you can remove the directory
# newdir.rmdir()
newfile= newdir / 'text_1.txt'
newfile.write_text("hello")
newfile.unlink()

## Module **argparse** <a name="argparse"></a>

This section follows the official Python documentation <http://docs.python.org>

Remember that Python is totally capable from reading from the user. There is a nice module to make a *friendly* user interaction.
The analog will be the *man* command in Linux or *help* command in Windows.

Some definitions:

+ Argument. It is a string in the command line passed by the shell to *execl()* or *execv()*. In Python, arguments are elements of
    *sys.argv[1:]*.
+ Option. It is an argument used to supply extra information to guide or customize the execution of the program. For instance *-v* or *-h*. However, some of these options can include whole words.
+ Option argument. It is an argument that follows an option. For instance *--file myfile.txt*.

Let's see one example

> **Note for the notebook:** `argparse` is meant for scripts run from the command line. Inside a notebook, `parser.parse_args()` would try to parse the kernel's own arguments, so the cell below saves the example as a script (`%%writefile`) and the cells after it run that script with different options.

In [ ]:
%%writefile argparse_demo.py
import argparse
parser = argparse.ArgumentParser()
parser.add_argument("-v", "--verbose", help="increase output verbosity",
                        action="store_true")
parser.add_argument("-c", "--lectures", type=str, help="provide the name of the lecture")
parser.add_argument("-o", "--other", type=int, default=10)
args = parser.parse_args()
if args.verbose:
    print("verbosity turned on")
if args.lectures:
    print(f"This is the {args.lectures} ")
    print(f"How many times {args.other}")

In [ ]:
!python argparse_demo.py -h

In [ ]:
!python argparse_demo.py -v -c "Advanced Python" -o 3

## Hashing <a name="hashing"></a>

*Hashing* refers to the mapping of an object to an integer, known as a *hash code*. We refer to an object as *hashable* if it satisfies the
following properties

+ The object can be compared for equality with other objects via the == operator
+ Whenever two objects compare as equal, they have the same has code.
+ The object hash code does not change during its lifetime.

In [1]:
a="BMIG 50303"
b="BMIG 50303"
c="UAMS"
hash(a),hash(b),hash(c)

(1292688767711790193, 1292688767711790193, 4296916526089099115)